In [1]:
"""
2. Structured Pruning — ResNet-50 / CIFAR-10
=============================================
Removes entire filters/channels (structured units) rather than
individual weights. Produces a physically smaller, denser model
that delivers REAL inference speedup without sparse runtimes.

Strategy: L1-norm filter pruning on Conv2d layers.
  - Rank each output filter by its L1 norm
  - Remove lowest-norm filters at each layer
  - Result: smaller conv weight tensors, fewer channels

Output: 2_structured_Pruning.json
"""

import os, json, time, copy, tempfile, warnings
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
BASELINE_MODEL_PATH   = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
OUTPUT_JSON           = "2_v2_structured_Pruning.json"

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE  = 128
NUM_WORKERS = 2
NUM_CLASSES = 10

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

# For structured pruning, pruning_ratios are per-layer fraction of filters to remove
PRUNING_RATIOS = [0.10] # , 0.20, 0.30, 0.40, 0.50
MAX_ACC_DROP   = 0.02
INFERENCE_RUNS = 50


# ── Model ─────────────────────────────────────────────────────────────────────
def build_model(num_classes=10):
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def load_baseline(path):
    model = build_model(NUM_CLASSES).to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    return model

# ── Data ──────────────────────────────────────────────────────────────────────
def get_test_loader():
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../data", train=False,
                                       download=True, transform=transform)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)


# ── Structured Pruning via torch.nn.utils.prune ───────────────────────────────
def apply_structured_pruning(model, ratio):
    """
    L1 structured pruning on all Conv2d layers:
    Remove `ratio` fraction of output filters (dim=0) ranked by L1 norm.
    Uses PyTorch's built-in ln_structured, then makes permanent.

    NOTE: PyTorch structured pruning zeroes entire filter rows but does NOT
    physically resize the weight tensors. For true size reduction the model
    needs to be rebuilt (see rebuild_pruned_model). Here we measure the
    theoretical sparsity and disk compression.
    """
    import torch.nn.utils.prune as prune_utils
    pruned = copy.deepcopy(model)

    for name, module in pruned.named_modules():
        if isinstance(module, nn.Conv2d) and module.weight.shape[0] > 1:
            n_to_prune = max(1, int(module.weight.shape[0] * ratio))
            # Clamp so we never prune all filters
            n_to_prune = min(n_to_prune, module.weight.shape[0] - 1)
            prune_utils.ln_structured(module, name="weight", amount=n_to_prune,
                                      n=1, dim=0)
            prune_utils.remove(module, "weight")

    return pruned


def structured_sparsity(model):
    """Fraction of all-zero filters in Conv2d layers."""
    zero_filters = total_filters = 0
    for _, m in model.named_modules():
        if isinstance(m, nn.Conv2d):
            w = m.weight.data
            norms = w.view(w.shape[0], -1).abs().sum(dim=1)
            zero_filters  += (norms == 0).sum().item()
            total_filters += w.shape[0]
    return zero_filters / total_filters if total_filters else 0.0

def weight_sparsity(model):
    zeros = total = 0
    for _, m in model.named_modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            zeros += (m.weight == 0).sum().item()
            total += m.weight.numel()
    return zeros / total if total else 0.0

def count_params(model):
    total  = sum(p.numel() for p in model.parameters())
    active = sum((p != 0).sum().item() for p in model.parameters())
    return total, active

def dense_size_mb(model):
    return sum(p.numel() for p in model.parameters()) * 4 / 1024**2

def sparse_size_mb(model):
    active = sum((p != 0).sum().item() for p in model.parameters())
    return active * 4 / 1024**2

def disk_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        size = os.path.getsize(tmp) / 1024**2
    finally:
        os.remove(tmp)
    return size


# ── Evaluation ────────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        preds.extend(model(inputs).argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }

def measure_gpu_ms(model):
    if DEVICE.type != "cuda": return -1.0
    model.eval()
    dummy = torch.randn(BATCH_SIZE, 3, 32, 32, device=DEVICE)
    with torch.no_grad():
        for _ in range(20): model(dummy)
        torch.cuda.synchronize()
        s = torch.cuda.Event(enable_timing=True)
        e = torch.cuda.Event(enable_timing=True)
        s.record()
        for _ in range(INFERENCE_RUNS): model(dummy)
        e.record()
        torch.cuda.synchronize()
    return s.elapsed_time(e) / INFERENCE_RUNS

def measure_cpu_ms(model):
    cpu_m = copy.deepcopy(model).cpu().eval()
    dummy = torch.randn(BATCH_SIZE, 3, 32, 32)
    with torch.no_grad():
        for _ in range(5): cpu_m(dummy)
        t0 = time.perf_counter()
        for _ in range(20): cpu_m(dummy)
    return (time.perf_counter() - t0) / 20 * 1000

from thop import profile

def compute_flops(model, device, input_size=(1, 3, 32, 32)):
    """
    Standard FLOPs computation using THOP.
    Returns:
        flops_total (int): total FLOPs (not MACs)
        flops_G (float): FLOPs in GFLOPs
    """
    model.eval()
    dummy = torch.randn(input_size).to(device)

    macs, _ = profile(model, inputs=(dummy,), verbose=False)
    flops = macs * 2  # convert MACs → FLOPs

    return int(flops), flops / 1e9

# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*60}")
    print("  2. Structured Pruning — ResNet-50 / CIFAR-10")
    print(f"  Device: {DEVICE}  |  Ratios: {[int(r*100) for r in PRUNING_RATIOS]}%")
    print(f"{'='*60}\n")

    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)

    model  = load_baseline(BASELINE_MODEL_PATH)
    loader = get_test_loader()

    base_dense = dense_size_mb(model)
    base_disk  = disk_size_mb(model)
    base_flops, base_flops_G = compute_flops(model, DEVICE)

    results       = []
    best_model    = None
    best_score    = -1.0
    best_ratio    = None

    for ratio in PRUNING_RATIOS:
        print(f"\n  Pruning {int(ratio*100)}% of filters per layer...")
        pruned = apply_structured_pruning(model, ratio)

        struct_sp = structured_sparsity(pruned)
        weight_sp = weight_sparsity(pruned)
        total_p, active_p = count_params(pruned)

        metrics  = evaluate(pruned, loader)
        inf_gpu  = measure_gpu_ms(pruned)
        inf_cpu  = measure_cpu_ms(pruned)
        acc_drop = baseline["accuracy"] - metrics["accuracy"]

        d_mb  = dense_size_mb(pruned)
        sp_mb = sparse_size_mb(pruned)
        dk_mb = disk_size_mb(pruned)

        pruned_flops, pruned_flops_G = compute_flops(pruned, DEVICE)

        row = {
            "filter_pruning_ratio"       : ratio,
            "structured_sparsity"        : round(struct_sp, 4),
            "weight_sparsity"            : round(weight_sp, 4),
            "accuracy"                   : round(metrics["accuracy"], 6),
            "precision"                  : round(metrics["precision"], 6),
            "recall"                     : round(metrics["recall"], 6),
            "f1"                         : round(metrics["f1"], 6),
            "accuracy_drop"              : round(acc_drop, 6),
            "params_total"               : total_p,
            "params_active"              : active_p,
            "size_dense_mb"              : round(d_mb, 4),
            "size_sparse_theoretical_mb" : round(sp_mb, 4),
            "size_disk_mb"               : round(dk_mb, 4),
            "disk_saved_mb"              : round(base_disk - dk_mb, 4),
            "inference_gpu_ms"           : round(inf_gpu, 4) if inf_gpu > 0 else None,
            "inference_cpu_ms"           : round(inf_cpu, 4),
            "flops_total"   : int(base_flops),
            "flops_active"  : pruned_flops,
            "flops_reduction_pct": round((1 - pruned_flops / base_flops) * 100, 2)
        }
        results.append(row)

        print(f"  Filter sparsity: {struct_sp*100:.1f}% | Weight sp: {weight_sp*100:.1f}% | "
              f"Acc: {metrics['accuracy']:.4f} (drop: {acc_drop:+.4f})")

        if acc_drop <= MAX_ACC_DROP:
            score = metrics["accuracy"] + ((base_disk - dk_mb) / base_disk) * 0.1
            if score > best_score:
                best_score, best_model, best_ratio = score, copy.deepcopy(pruned), ratio

    report = {
        "method"             : "structured_pruning",
        "description"        : "L1 structured filter pruning — entire Conv2d output filters removed",
        "pruning_granularity": "filter (output channel, dim=0)",
        "baseline"           : baseline,
        "baseline_size_breakdown": {
            "dense_ram_mb": round(base_dense, 4),
            "disk_pth_mb" : round(base_disk, 4),
        },
        "pruning_criterion"  : "L1 norm of filter (sum of |w| per output filter)",
        "device"             : str(DEVICE),
        "max_acc_drop_threshold": MAX_ACC_DROP,
        "best_ratio"         : best_ratio,
        "baseline_flops": int(base_flops),
        "results"            : results,
        "notes": {
            "structured_sparsity": "Fraction of zero-norm filters in Conv2d layers",
            "weight_sparsity"    : "Fraction of zero weight values (all weights in zero filters)",
            "real_speedup"       : "Structured pruning CAN yield real speedup after model rebuild — zero-row Conv ops are skipped by cuDNN. PyTorch's prune API zeroes rows but keeps tensor shapes; rebuild for actual FLOP reduction.",
            "size_note"          : "Disk size compresses well due to zero-filter rows",
            "flops_total"  : "Dense FLOPs — 2*Cin*Cout*kH*kW*oH*oW per conv layer",
            "flops_active" : "FLOPs * density — theoretical sparse compute savings"
        }
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)
    print(f"\n  ✓ Saved -> {OUTPUT_JSON}")


if __name__ == "__main__":
    main()



  2. Structured Pruning — ResNet-50 / CIFAR-10
  Device: cuda  |  Ratios: [10]%


  Pruning 10% of filters per layer...
  Filter sparsity: 9.9% | Weight sp: 9.9% | Acc: 0.9322 (drop: -0.0002)

  ✓ Saved -> 2_v2_structured_Pruning.json
